# 05 · Ablation — 컴포넌트 분해 (논문 필수)

기여 3개(carry / BiMamba / MOSAIC)를 **각각 분리**하는 사다리. 이게 없으면 Intro의
"BiMamba가 +[XX]p", "MOSAIC이 경계 떨림 제거" 주장을 뒷받침할 근거가 없다.

| 태그 | carry | BiMamba | MOSAIC | 역할 |
|---|---|---|---|---|
| `acm` | X | X | X | 바닥 (plain Mamba dec) |
| `acm_carry` | O | X | X | carry 단독 |
| `acm_bimamba` | O | O | X | +BiMamba (성능 축) |
| `acm_s7` | O | X | O | +MOSAIC (BiMamba 없이 -- 직교성) |
| **`ours`** | O | O | O | **최종** |

`acm` / `ours` 는 메인 표(01·02)에서 이미 학습·평가됨 -> 여기선 **가운데 3개만** 추가.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

TASK  = cf.MAIN_SIM              # 'insertion'
SEEDS = cf.MAIN_SEEDS            # [0,1,2,3] — 메인과 동일
REPS  = list(range(cf.EVAL_REPEATS))   # 150k ckpt x 5회 반복 (메인과 동일 프로토콜)
N_EP  = cf.EVAL_N_EP
NGPU  = 8

print('ablation ladder:', cf.ABLATION)
print('여기서 추가 학습:', cf.ABLATION_TRAIN, '| task:', TASK, '| seeds:', SEEDS)
print('학습:', f'{cf.STEPS:,} step (lr 고정)', '| eval:', f'{cf.CKPT_STEP:,} ckpt x {len(REPS)} reps')
print('추가 학습 잡:', len(cf.ABLATION_TRAIN) * len(SEEDS))

## 학습 — 추가 3모델 x seed (GPU 청크)

In [ ]:
jobs = [(t, s, TASK) for s in SEEDS for t in cf.ABLATION_TRAIN]
for i in range(0, len(jobs), NGPU):
    chunk = jobs[i:i + NGPU]
    print('\n===== %d 잡 =====' % len(chunk))
    for j in chunk:
        print('  ', j)
    cf.launch_training_live(chunk)
print('\nablation 학습 완료')

In [ ]:
jobs = [(t, s, r) for s in SEEDS for t in cf.ABLATION_TRAIN for r in REPS]
todo = [(t, s, r) for (t, s, r) in jobs if cf.rep_sr(t, s, TASK, r) is None]
print(f'전체 {len(jobs)} run / 남은 {len(todo)} run')

for i in range(0, len(todo), NGPU):
    chunk = todo[i:i + NGPU]
    labeled = []
    for g, (t, s, r) in enumerate(chunk):
        try:
            labeled.append((f'{t}/seed{s}/rep{r}',
                            cf.repeat_eval_cmd(t, s, r, task=TASK, gpu_id=g, n_episodes=N_EP)))
        except FileNotFoundError as e:
            print('  skip:', t, s, r, e)
    if labeled:
        cf.launch_cmds_live(labeled)
print('\nablation eval 완료')

In [ ]:
N_EP, SELECT = 50, 'last'
labeled = []
for s in SEEDS:
    for t in cf.ABLATION_TRAIN:
        try:
            labeled.append((f'{t}/seed{s}',
                cf.make_eval_cmd(t, seed=s, task=TASK, gpu_id=len(labeled) % NGPU,
                                 n_episodes=N_EP, select=SELECT)))
        except FileNotFoundError as e:
            print('  skip:', t, s, e)
for i in range(0, len(labeled), NGPU):
    cf.launch_cmds_live(labeled[i:i + NGPU])

## Ablation 표 (SR + 경계 떨림) -> `ablation.csv`
SR = 150k ckpt × 5 rep × seed (mean±std). **각 행이 이전 행 대비 무엇을 더했는지**가 곧 기여.

In [ ]:
import csv
import numpy as np
import smooth_metrics as sm

fps, K = cf.fps_of(TASK), 100
rows = []
for t in cf.ABLATION:
    agg = cf.sr_over_reps(t, task=TASK, seeds=SEEDS, reps=REPS)
    trajs = []
    for s in SEEDS:
        trajs += cf.action_trajs(t, s, TASK, reps=REPS)      # rep 5개 pool
    m = sm.aggregate_smoothness(trajs, chunk=K, fps=fps) if trajs else {}
    g = lambda k, d=4: (f'%.{d}f' % m[k]) if k in m else '-'
    rows.append({'tag': t, 'model': cf.v23.MODEL_LABELS.get(t, t),
                 'SR': ('%.1f' % agg['mean']) if agg['mean'] is not None else '-',
                 'SR_std': ('%.1f' % agg['std']) if agg['mean'] is not None else '-',
                 'n_run': agg['n_runs'], 'n_traj': len(trajs),
                 'boundary_jerk': g('boundary_jerk'), 'interior_jerk': g('interior_jerk'),
                 'contrast': g('boundary_jerk_contrast'), 'SPARC': g('sparc', 3)})

out = cf.OUTPUT_BASE / 'ablation'
out.mkdir(parents=True, exist_ok=True)
with open(out / 'ablation.csv', 'w', newline='', encoding='utf-8') as fh:
    w = csv.DictWriter(fh, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)

hdr = f"{'MODEL':<34}{'SR':>14}{'b-jerk':>9}{'i-jerk':>9}{'contrast':>10}{'SPARC':>8}"
print(hdr)
print('-' * len(hdr))
for r in rows:
    sr = f"{r['SR']} ± {r['SR_std']}" if r['SR'] != '-' else '-'
    print(f"{r['model']:<34}{sr:>14}{r['boundary_jerk']:>9}{r['interior_jerk']:>9}"
          f"{r['contrast']:>10}{r['SPARC']:>8}")
print('\n저장:', out / 'ablation.csv')